# W86 Frontier-Scale Closure on Vertex AI Colab Enterprise (NVIDIA L4)

Closes the post-W83 P0 substrate blockers (#25, #26, the #27 hidden-state-intercept-at-32k axis) by loading **Llama-3.1-8B-Instruct** in bf16 on a real GPU and running the W80 instrumentation contract end-to-end.

**Inputs (from Vertex AI Secret Manager):**
- `coordpy-hf-token` — HuggingFace read token with `meta-llama/Llama-3.1-8B-Instruct` access

**Inputs (from GCS):**
- `gs://coordpy-frontier-closure-1779235853/repo_snapshot.tar.gz` — frozen snapshot of the CoordPy repo with the W86 closure code, uploaded by the orchestrator. The notebook never reaches out to GitHub.

**Outputs (to GCS):**
- `gs://coordpy-frontier-closure-1779235853/w86_<TIMESTAMP>/frontier_closure_report.json`
- per-step sidecar JSON files in the same prefix

**Anti-cheat preserved:**
- the closure code lives in the `coordpy` package; the notebook only invokes it.
- the HF token never leaves Secret Manager; the notebook reads it just-in-time.
- nothing here computes the report; everything is delegated to `scripts/run_frontier_closure_w86.py`.

In [ ]:
# --- 1. Environment probe ---
import os, sys, subprocess, json, datetime
print('python:', sys.version)
print('cwd:', os.getcwd())
RUN_TS = datetime.datetime.now(
    tz=datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('RUN_TS:', RUN_TS)
OUT_DIR = f'/home/jupyter/w86_{RUN_TS}'
os.makedirs(OUT_DIR, exist_ok=True)
print('OUT_DIR:', OUT_DIR)
BUCKET = 'gs://coordpy-frontier-closure-1779235853'
GCS_DEST = f'{BUCKET}/w86_{RUN_TS}/'
print('GCS_DEST:', GCS_DEST)

In [ ]:
# --- 2. GPU probe ---
subprocess.run(['nvidia-smi'], check=False)
import torch
print('torch:', torch.__version__,
      'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  '
          f'mem={p.total_memory / (1024**3):.2f} GiB  '
          f'sm={p.major}.{p.minor}')

In [ ]:
# --- 3. Install transformers / accelerate / secret manager / numpy ---
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0',
    'accelerate>=0.34.0',
    'huggingface_hub>=0.24.0',
    'google-cloud-secret-manager>=2.20.0',
    'numpy>=1.24',
], check=True)
import transformers, accelerate, huggingface_hub
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)
print('huggingface_hub:', huggingface_hub.__version__)

In [ ]:
# --- 4. Fetch HF token from Secret Manager and log into HF Hub ---
from google.cloud import secretmanager  # type: ignore
PROJECT_ID = 'gen-lang-client-0387794233'
SECRET_ID = 'coordpy-hf-token'
client = secretmanager.SecretManagerServiceClient()
name = f'projects/{PROJECT_ID}/secrets/{SECRET_ID}/versions/latest'
resp = client.access_secret_version(request={'name': name})
hf_token = resp.payload.data.decode('utf-8').strip()
assert hf_token.startswith('hf_'), (
    'HF token must start with hf_')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print('HF login OK; token len =', len(hf_token))

In [ ]:
# --- 5. Download repo snapshot from GCS and pip-install coordpy in editable mode ---
subprocess.run([
    'gsutil', 'cp', f'{BUCKET}/repo_snapshot.tar.gz',
    '/home/jupyter/repo_snapshot.tar.gz',
], check=True)
os.makedirs('/home/jupyter/context-zero', exist_ok=True)
subprocess.run([
    'tar', '-xzf', '/home/jupyter/repo_snapshot.tar.gz',
    '-C', '/home/jupyter/context-zero',
    '--strip-components=1',
], check=True)
os.chdir('/home/jupyter/context-zero')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
], check=True)
import coordpy
print('coordpy:', coordpy.__version__,
      'SDK:', getattr(coordpy, 'SDK_VERSION', '?'))

In [ ]:
# --- 6. Run the closure driver (all three issues) ---
rc = subprocess.run([
    sys.executable,
    'scripts/run_frontier_closure_w86.py',
    '--out-dir', OUT_DIR,
    '--model-name', 'meta-llama/Llama-3.1-8B-Instruct',
    '--device', 'cuda:0',
    '--precision-tier', 'tier_bf16',
    '--seed', '86001001',
    '--n-train-prompts', '40',
    '--n-eval-prompts', '10',
    '--horizon-tokens', '32768',
], check=False)
print('closure driver exit code:', rc.returncode)

In [ ]:
# --- 7. Print the report summary ---
with open(os.path.join(OUT_DIR, 'frontier_closure_report.json'),
          'rb') as fh:
    report = json.load(fh)
print('report_cid:', report.get('report_cid'))
for k in ('closure_25', 'closure_26', 'closure_27'):
    c = report.get(k, {})
    if not c:
        continue
    print(f'\n=== {k} ===')
    for kk in (
            'conformance', 'hidden_state_intercept_bench',
            'replay_from_kv', 'w83_load_bearing_claim_reproduced',
            'live_strictly_beats_synthetic',
            'intercept_moves_cid_at_min_32k',
            'wall_seconds', 'wall_seconds_conformance',
            'wall_seconds_replay', 'wall_seconds_hidden_state_intercept',
            'error',
    ):
        if kk in c:
            v = c[kk]
            if isinstance(v, dict):
                print(f'  {kk}: {json.dumps(v)[:400]}')
            else:
                print(f'  {kk}: {v}')

In [ ]:
# --- 8. Upload results to GCS ---
rc = subprocess.run([
    'gsutil', '-m', 'cp', '-r',
    f'{OUT_DIR}/.', GCS_DEST,
], check=False)
print('gsutil exit:', rc.returncode)
print('uploaded to', GCS_DEST)
subprocess.run(['gsutil', 'ls', '-l', GCS_DEST],
               check=False)